# 20 - Train Sevilla Mention Sentiment ABSA Model v1

Este notebook entrena el modelo de sentimiento por mención para Hidden Gems Sevilla.

Objetivo:

```text
mención de plato + contexto de reseña
→ sentimiento hacia ese plato concreto
```

Clases:

negative
neutral
positive

Este modelo no clasifica el sentimiento general del restaurante ni de la reseña completa. Clasifica el sentimiento hacia la mención concreta del plato.

Ejemplo:

"Las croquetas estaban increíbles, pero el arroz fue decepcionante."
croquetas → positive
arroz → negative

El modelo se utilizará después de:

Hybrid + NER v2
→ Normalización / Entity Linking
→ ABSA sentiment model
→ señales local-plato v2
→ ranking Hidden Gems v2

In [1]:
# ============================================================
# 01. Instalación e imports
# ============================================================

!pip -q install -U transformers datasets accelerate evaluate scikit-learn

import os
import re
import json
import math
import random
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    classification_report,
    confusion_matrix,
)

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 79.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 100.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 104.5 MB/s eta 0:00:0000:01
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
# ============================================================
# 02. Configuración general
# ============================================================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

NOTEBOOK_NAME = "20_train_sevilla_mention_sentiment_absa_v1"
VERSION = "sevilla_mention_sentiment_absa_beto_v1"

MODEL_CHECKPOINT = "dccuchile/bert-base-spanish-wwm-cased"

OUTPUT_ROOT = Path("/kaggle/working")
MODEL_OUTPUT_DIR = OUTPUT_ROOT / VERSION
REPORTS_DIR = OUTPUT_ROOT / "reports"
PREDICTIONS_DIR = OUTPUT_ROOT / "predictions"

for path in [MODEL_OUTPUT_DIR, REPORTS_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LABEL_LIST = ["negative", "neutral", "positive"]
LABEL2ID = {label: idx for idx, label in enumerate(LABEL_LIST)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

MAX_LENGTH = 256

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
NUM_EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08
LABEL_SMOOTHING = 0.03

USE_FP16 = bool(torch.cuda.is_available())

CONFIG = {
    "notebook": NOTEBOOK_NAME,
    "version": VERSION,
    "model_checkpoint": MODEL_CHECKPOINT,
    "labels": LABEL_LIST,
    "max_length": MAX_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "label_smoothing": LABEL_SMOOTHING,
    "seed": SEED,
    "fp16": USE_FP16,
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

{
  "notebook": "20_train_sevilla_mention_sentiment_absa_v1",
  "version": "sevilla_mention_sentiment_absa_beto_v1",
  "model_checkpoint": "dccuchile/bert-base-spanish-wwm-cased",
  "labels": [
    "negative",
    "neutral",
    "positive"
  ],
  "max_length": 256,
  "train_batch_size": 8,
  "eval_batch_size": 16,
  "gradient_accumulation_steps": 2,
  "epochs": 5,
  "learning_rate": 2e-05,
  "weight_decay": 0.01,
  "warmup_ratio": 0.08,
  "label_smoothing": 0.03,
  "seed": 42,
  "fp16": true
}


In [3]:
# ============================================================
# 03. Localización automática del dataset
# Prioridad: extended JSONL > extended CSV > original
# ============================================================

def find_input_file_priority(filename_patterns: List[str]) -> Path:
    search_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]

    for pattern in filename_patterns:
        candidates = []
        for root in search_roots:
            if root.exists():
                candidates.extend(root.rglob(pattern))

        if candidates:
            candidates = sorted(candidates, key=lambda p: str(p))
            return candidates[0]

    raise FileNotFoundError(
        f"No se encontró ningún archivo con patrones: {filename_patterns}. "
        "Revisa que hayas añadido el dataset privado de Kaggle."
    )


DATA_PATH = find_input_file_priority([
    "sevilla_mention_sentiment_annotation_dataset_v1_extended.jsonl",
])

print("Dataset encontrado:", DATA_PATH)

if "extended" not in DATA_PATH.name:
    raise ValueError(
        f"Se ha cargado un dataset NO extendido: {DATA_PATH.name}. "
        "Para el entrenamiento fuerte debe usarse el dataset extendido."
    )

Dataset encontrado: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-sevilla-v1/sevilla_mention_sentiment_annotation_dataset_v1_extended.jsonl


In [4]:
# ============================================================
# 04. Carga y validación
# ============================================================

def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


if DATA_PATH.suffix.lower() == ".jsonl":
    df = read_jsonl(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head(3))
print(df.columns.tolist())

if len(df) < 5000:
    raise ValueError(
        f"El dataset parece no ser el extendido. Filas cargadas: {len(df)}. "
        "El extendido debería tener aproximadamente 5.200 filas."
    )

print("Columnas disponibles:")
for c in df.columns:
    print("-", c)

Shape: (5200, 23)


,place_id,place_name,dish_id,dish_name,dish_display_name,review_id,rating_value,weak_sentiment_label,weak_sentiment_score,weak_sentiment_confidence,...,context_sentence,target_clause_context,review_text_raw,annotation_id,dish_mention,model_input_text,manual_sentiment_label,manual_sentiment_confidence,manual_sentiment_notes,annotation_status
0,db08a79e-1470-4e96-adcb-2165e6f06a2a,Restaurante La Montanera (Los Remedios),b31eaf94-9043-4231-8018-450979e974c4,presa iberica,None,0066d3ed-021b-4630-b205-90efcbb3108a,5.0,positive,1.6709,0.86,...,La presa muy buena aunque demasiado poco hecha...,La presa muy buena aunque demasiado poco hecha...,Restaurante familiar con gran trayectoria en l...,sevilla_sentiment_000001,presa,[REVIEW] Restaurante familiar con gran trayect...,None,NaN,None,pending
1,db08a79e-1470-4e96-adcb-2165e6f06a2a,Restaurante La Montanera (Los Remedios),bc321565-14c5-49d7-b43a-8d1fc3ae5f31,ensaladilla,None,0066d3ed-021b-4630-b205-90efcbb3108a,5.0,positive,1.1500,0.38,...,El cóctel de marisco nunca defrauda y la ensal...,El cóctel de marisco nunca defrauda y la ensal...,Restaurante familiar con gran trayectoria en l...,sevilla_sentiment_000002,ensaladilla,[REVIEW] Restaurante familiar con gran trayect...,None,NaN,None,pending
2,a103f56a-fe4e-4c50-be8a-f454f04e9ede,Ena Sevilla,d82d9fb6-1a16-4a95-9931-3b8262c5a9ab,croqueta,None,0076f60e-6a91-4591-af9d-1479914fdcfd,1.0,negative,-1.1500,0.36,...,"Una vez que nos atendieron, dos cervezas, dos ...","Una vez que nos atendieron, dos cervezas, dos ...","Desgraciadamente, comparto con otras reseñas l...",sevilla_sentiment_000003,croquetas,"[REVIEW] Desgraciadamente, comparto con otras ...",None,NaN,None,pending


['place_id', 'place_name', 'dish_id', 'dish_name', 'dish_display_name', 'review_id', 'rating_value', 'weak_sentiment_label', 'weak_sentiment_score', 'weak_sentiment_confidence', 'weak_sentiment_reliability_tier', 'weak_sentiment_reason', 'mention_text', 'context_sentence', 'target_clause_context', 'review_text_raw', 'annotation_id', 'dish_mention', 'model_input_text', 'manual_sentiment_label', 'manual_sentiment_confidence', 'manual_sentiment_notes', 'annotation_status']
Columnas disponibles:
- place_id
- place_name
- dish_id
- dish_name
- dish_display_name
- review_id
- rating_value
- weak_sentiment_label
- weak_sentiment_score
- weak_sentiment_confidence
- weak_sentiment_reliability_tier
- weak_sentiment_reason
- mention_text
- context_sentence
- target_clause_context
- review_text_raw
- annotation_id
- dish_mention
- model_input_text
- manual_sentiment_label
- manual_sentiment_confidence
- manual_sentiment_notes
- annotation_status


In [5]:
# ============================================================
# 04B. Detección robusta de columnas
# ============================================================

def first_existing(possible_cols: List[str], required: bool = True) -> Optional[str]:
    for col in possible_cols:
        if col in df.columns:
            return col
    if required:
        raise KeyError(f"No se encontró ninguna columna entre: {possible_cols}")
    return None


COL_ANNOTATION_ID = first_existing(["annotation_id", "mention_id", "id"], required=False)
COL_REVIEW_ID = first_existing(["review_id"], required=False)
COL_PLACE_ID = first_existing(["place_id"], required=False)
COL_PLACE_NAME = first_existing(["place_name"], required=False)

COL_MENTION = first_existing([
    "dish_mention_text",
    "mention_text",
    "selected_mention_text_v2",
])

COL_DISH_NAME = first_existing([
    "display_dish_name_es_v1",
    "canonical_dish_name_es_v1",
    "normalized_dish_name_es_v1",
    "dish_display_name",
    "dish_name",
    "current_dish_display_name",
], required=False)

COL_CONTEXT = first_existing([
    "context_sentence",
    "target_clause_context",
    "window_context",
], required=False)

COL_WINDOW = first_existing([
    "window_context",
    "context_window",
    "text_window",
], required=False)

COL_REVIEW_TEXT = first_existing([
    "review_text_raw",
    "text",
    "review_text",
], required=False)

COL_RATING = first_existing([
    "rating_value",
    "review_rating",
    "stars",
], required=False)

COL_WEAK_LABEL = first_existing([
    "weak_sentiment_label",
    "mention_sentiment_label_v1",
    "current_sentiment_label",
], required=False)

COL_MANUAL_LABEL = first_existing([
    "manual_sentiment_label",
    "manual_label",
    "gold_sentiment_label",
], required=False)

COL_ANNOTATION_STATUS = first_existing([
    "annotation_status",
    "source_type",
], required=False)

print("Column mapping:")
column_mapping = {
    "annotation_id": COL_ANNOTATION_ID,
    "review_id": COL_REVIEW_ID,
    "place_id": COL_PLACE_ID,
    "place_name": COL_PLACE_NAME,
    "mention": COL_MENTION,
    "dish_name": COL_DISH_NAME,
    "context": COL_CONTEXT,
    "window": COL_WINDOW,
    "review_text": COL_REVIEW_TEXT,
    "rating": COL_RATING,
    "weak_label": COL_WEAK_LABEL,
    "manual_label": COL_MANUAL_LABEL,
    "annotation_status": COL_ANNOTATION_STATUS,
}
print(json.dumps(column_mapping, indent=2, ensure_ascii=False))

Column mapping:
{
  "annotation_id": "annotation_id",
  "review_id": "review_id",
  "place_id": "place_id",
  "place_name": "place_name",
  "mention": "mention_text",
  "dish_name": "dish_display_name",
  "context": "context_sentence",
  "window": null,
  "review_text": "review_text_raw",
  "rating": "rating_value",
  "weak_label": "weak_sentiment_label",
  "manual_label": "manual_sentiment_label",
  "annotation_status": "annotation_status"
}


In [6]:
# ============================================================
# 05. Limpieza de etiquetas y gold label
# ============================================================

VALID_LABELS = set(LABEL_LIST)

def clean_label(value: Any) -> Optional[str]:
    if value is None:
        return None

    if isinstance(value, float) and math.isnan(value):
        return None

    text = str(value).strip().lower()

    if text in {"", "nan", "none", "null", "na"}:
        return None

    mapping = {
        "pos": "positive",
        "positivo": "positive",
        "positiva": "positive",
        "positive": "positive",
        "neu": "neutral",
        "neutro": "neutral",
        "neutra": "neutral",
        "neutral": "neutral",
        "neg": "negative",
        "negativo": "negative",
        "negativa": "negative",
        "negative": "negative",
    }

    return mapping.get(text, None)


def determine_gold_label(row: pd.Series) -> Tuple[Optional[str], str, float]:
    manual = clean_label(row.get(COL_MANUAL_LABEL)) if COL_MANUAL_LABEL else None
    weak = clean_label(row.get(COL_WEAK_LABEL)) if COL_WEAK_LABEL else None

    annotation_status = str(row.get(COL_ANNOTATION_STATUS, "") or "").lower()

    # Etiqueta manual/gold tiene prioridad.
    if manual in VALID_LABELS:
        if annotation_status.startswith("manual") or "augmented" in annotation_status:
            return manual, "manual_gold", 2.0
        return manual, "manual_gold", 2.0

    # Si no hay manual, usamos weak label con menor peso.
    if weak in VALID_LABELS:
        return weak, "weak_hybrid", 0.75

    return None, "unlabeled", 0.0


gold_rows = []
for _, row in df.iterrows():
    label, label_source, sample_weight = determine_gold_label(row)
    gold_rows.append({
        "gold_label": label,
        "label_source": label_source,
        "sample_weight": sample_weight,
    })

gold_df = pd.DataFrame(gold_rows)
df = pd.concat([df.reset_index(drop=True), gold_df], axis=1)

df["label_id"] = df["gold_label"].map(LABEL2ID)

usable_df = df[df["gold_label"].isin(LABEL_LIST)].copy().reset_index(drop=True)

print("Usable rows:", usable_df.shape)
print("\nGold label distribution:")
display(usable_df["gold_label"].value_counts())

print("\nLabel source distribution:")
display(usable_df["label_source"].value_counts())

display(usable_df[[COL_MENTION, "gold_label", "label_source"]].head(10))

Usable rows: (5200, 27)

Gold label distribution:


gold_label
positive    3105
negative    1104
neutral      991
Name: count, dtype: int64


Label source distribution:


label_source
weak_hybrid    2911
manual_gold    2289
Name: count, dtype: int64

,mention_text,gold_label,label_source
0,presa,positive,weak_hybrid
1,ensaladilla,positive,weak_hybrid
2,croquetas,negative,weak_hybrid
3,solomillo,negative,weak_hybrid
4,patatas bravas,positive,weak_hybrid
5,risotto,positive,weak_hybrid
6,tarta de queso,positive,weak_hybrid
7,helado,positive,weak_hybrid
8,torrija,positive,weak_hybrid
9,empanadas,positive,weak_hybrid


In [7]:
# ============================================================
# 06. Construcción del input ABSA
# ============================================================

def safe_text(row: pd.Series, col: Optional[str]) -> str:
    if col is None:
        return ""
    value = row.get(col, "")
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    return str(value).strip()


def build_model_input(row: pd.Series) -> str:
    mention = safe_text(row, COL_MENTION)
    dish = safe_text(row, COL_DISH_NAME)
    context = safe_text(row, COL_CONTEXT)
    window = safe_text(row, COL_WINDOW)
    review_text = safe_text(row, COL_REVIEW_TEXT)
    place = safe_text(row, COL_PLACE_NAME)
    rating = safe_text(row, COL_RATING)

    parts = [
        f"Mención: {mention}",
    ]

    if dish and dish.lower() != mention.lower():
        parts.append(f"Plato normalizado: {dish}")

    if context:
        parts.append(f"Contexto local: {context}")

    if window and window != context:
        parts.append(f"Ventana: {window}")

    # Solo añadimos review completa si no existe contexto local.
    # Evitamos que el modelo dependa demasiado de sentimiento global.
    if not context and review_text:
        parts.append(f"Review: {review_text[:700]}")

    if rating:
        parts.append(f"Rating: {rating}")

    if place:
        parts.append(f"Local: {place}")

    return " [SEP] ".join(parts)


usable_df["model_input"] = usable_df.apply(build_model_input, axis=1)
usable_df["input_length_chars"] = usable_df["model_input"].map(len)

print("Input length:")
display(usable_df["input_length_chars"].describe())

display(usable_df[["model_input", "gold_label", "label_source"]].head(5))

Input length:


count    5200.000000
mean      202.727308
std        79.763443
min        97.000000
25%       160.000000
50%       177.000000
75%       215.000000
max       933.000000
Name: input_length_chars, dtype: float64

,model_input,gold_label,label_source
0,Mención: presa [SEP] Contexto local: La presa ...,positive,weak_hybrid
1,Mención: ensaladilla [SEP] Contexto local: El ...,positive,weak_hybrid
2,Mención: croquetas [SEP] Contexto local: Una v...,negative,weak_hybrid
3,Mención: solomillo [SEP] Contexto local: La co...,negative,weak_hybrid
4,Mención: patatas bravas [SEP] Contexto local: ...,positive,weak_hybrid


In [8]:
# ============================================================
# 07. Split train/validation/test agrupado por review_id
# ============================================================

# Creamos un group_id robusto para evitar fuga de datos:
# si existe review_id, agrupamos por review_id;
# si falta en alguna fila, usamos un identificador único por fila.
fallback_group_id = pd.Series(
    usable_df.index.astype(str),
    index=usable_df.index,
)

if COL_REVIEW_ID and COL_REVIEW_ID in usable_df.columns:
    usable_df["group_id"] = (
        usable_df[COL_REVIEW_ID]
        .astype("string")
        .fillna(fallback_group_id)
        .astype(str)
    )
else:
    usable_df["group_id"] = fallback_group_id.astype(str)

# Primer split: train vs temp
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED,
)

train_idx, temp_idx = next(
    gss.split(
        usable_df,
        groups=usable_df["group_id"],
    )
)

train_df = usable_df.iloc[train_idx].copy().reset_index(drop=True)
temp_df = usable_df.iloc[temp_idx].copy().reset_index(drop=True)

# Segundo split: validation vs test
gss2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=SEED,
)

val_idx, test_idx = next(
    gss2.split(
        temp_df,
        groups=temp_df["group_id"],
    )
)

val_df = temp_df.iloc[val_idx].copy().reset_index(drop=True)
test_df = temp_df.iloc[test_idx].copy().reset_index(drop=True)

split_summary = {
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "train_labels": train_df["gold_label"].value_counts().to_dict(),
    "validation_labels": val_df["gold_label"].value_counts().to_dict(),
    "test_labels": test_df["gold_label"].value_counts().to_dict(),
    "train_label_source": train_df["label_source"].value_counts().to_dict(),
    "validation_label_source": val_df["label_source"].value_counts().to_dict(),
    "test_label_source": test_df["label_source"].value_counts().to_dict(),
    "train_review_groups": int(train_df["group_id"].nunique()),
    "validation_review_groups": int(val_df["group_id"].nunique()),
    "test_review_groups": int(test_df["group_id"].nunique()),
}

print(json.dumps(split_summary, indent=2, ensure_ascii=False))

{
  "train_rows": 4155,
  "validation_rows": 534,
  "test_rows": 511,
  "train_labels": {
    "positive": 2469,
    "negative": 887,
    "neutral": 799
  },
  "validation_labels": {
    "positive": 330,
    "negative": 110,
    "neutral": 94
  },
  "test_labels": {
    "positive": 306,
    "negative": 107,
    "neutral": 98
  },
  "train_label_source": {
    "weak_hybrid": 2327,
    "manual_gold": 1828
  },
  "validation_label_source": {
    "weak_hybrid": 307,
    "manual_gold": 227
  },
  "test_label_source": {
    "weak_hybrid": 277,
    "manual_gold": 234
  },
  "train_review_groups": 2992,
  "validation_review_groups": 374,
  "test_review_groups": 375
}


In [9]:
# ============================================================
# 08. Conversión a Dataset
# ============================================================

def dataframe_to_records(dataframe: pd.DataFrame) -> List[Dict[str, Any]]:
    records = []

    meta_cols = [
        COL_ANNOTATION_ID,
        COL_REVIEW_ID,
        COL_PLACE_ID,
        COL_PLACE_NAME,
        COL_MENTION,
        COL_DISH_NAME,
        COL_CONTEXT,
        COL_RATING,
    ]

    for _, row in dataframe.iterrows():
        record = {
            "model_input": str(row["model_input"]),
            "label": int(row["label_id"]),
            "sample_weight": float(row["sample_weight"]),
            "gold_label": str(row["gold_label"]),
            "label_source": str(row["label_source"]),
        }

        for col in meta_cols:
            if col and col in dataframe.columns:
                value = row.get(col)
                if isinstance(value, float) and math.isnan(value):
                    value = None
                record[col] = None if value is None else str(value)

        records.append(record)

    return records


dataset = DatasetDict({
    "train": Dataset.from_list(dataframe_to_records(train_df)),
    "validation": Dataset.from_list(dataframe_to_records(val_df)),
    "test": Dataset.from_list(dataframe_to_records(test_df)),
})

dataset

DatasetDict({
    train: Dataset({
        features: ['model_input', 'label', 'sample_weight', 'gold_label', 'label_source', 'annotation_id', 'review_id', 'place_id', 'place_name', 'mention_text', 'dish_display_name', 'context_sentence', 'rating_value'],
        num_rows: 4155
    })
    validation: Dataset({
        features: ['model_input', 'label', 'sample_weight', 'gold_label', 'label_source', 'annotation_id', 'review_id', 'place_id', 'place_name', 'mention_text', 'dish_display_name', 'context_sentence', 'rating_value'],
        num_rows: 534
    })
    test: Dataset({
        features: ['model_input', 'label', 'sample_weight', 'gold_label', 'label_source', 'annotation_id', 'review_id', 'place_id', 'place_name', 'mention_text', 'dish_display_name', 'context_sentence', 'rating_value'],
        num_rows: 511
    })
})

In [10]:
# ============================================================
# 09. Tokenización
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

def tokenize_batch(batch: Dict[str, List[Any]]) -> Dict[str, Any]:
    tokenized = tokenizer(
        batch["model_input"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    tokenized["labels"] = batch["label"]
    tokenized["sample_weight"] = batch["sample_weight"]

    return tokenized


tokenized_dataset = dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

tokenized_dataset

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Map:   0%|          | 0/4155 [00:00<?, ? examples/s]

Map:   0%|          | 0/534 [00:00<?, ? examples/s]

Map:   0%|          | 0/511 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sample_weight', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4155
    })
    validation: Dataset({
        features: ['sample_weight', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 534
    })
    test: Dataset({
        features: ['sample_weight', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 511
    })
})

In [11]:
# ============================================================
# 10. Métricas
# ============================================================

def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(logits)
    return exp / np.sum(exp, axis=-1, keepdims=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax_np(logits)
    preds = probs.argmax(axis=-1)

    accuracy = accuracy_score(labels, preds)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0,
    )

    per_class_precision, per_class_recall, per_class_f1, per_class_support = precision_recall_fscore_support(
        labels,
        preds,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )

    metrics = {
        "accuracy": accuracy,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "macro_f1": f1_macro,
        "weighted_precision": precision_weighted,
        "weighted_recall": recall_weighted,
        "weighted_f1": f1_weighted,
    }

    for idx, label_name in ID2LABEL.items():
        metrics[f"{label_name}_precision"] = per_class_precision[idx]
        metrics[f"{label_name}_recall"] = per_class_recall[idx]
        metrics[f"{label_name}_f1"] = per_class_f1[idx]
        metrics[f"{label_name}_support"] = int(per_class_support[idx])

    return metrics

In [12]:
# ============================================================
# 11. Pesos de clase
# ============================================================

def compute_class_weights(dataframe: pd.DataFrame) -> torch.Tensor:
    counts = dataframe["label_id"].value_counts().reindex([0, 1, 2], fill_value=0).astype(float)
    total = counts.sum()

    freqs = counts / max(total, 1)
    weights = 1.0 / np.sqrt(np.maximum(freqs, 1e-8))

    # Normalizamos media 1 y capamos para estabilidad.
    weights = weights / weights.mean()
    weights = np.clip(weights, 0.5, 2.5)

    print("Class counts:")
    for idx, count in counts.items():
        print(ID2LABEL[idx], int(count))

    print("\nClass weights:")
    for idx, weight in enumerate(weights):
        print(ID2LABEL[idx], round(float(weight), 4))

    return torch.tensor(weights, dtype=torch.float)


CLASS_WEIGHTS = compute_class_weights(train_df)

Class counts:
negative 887
neutral 799
positive 2469

Class weights:
negative 1.1308
neutral 1.1914
positive 0.6778


In [13]:
# ============================================================
# 12. Trainer con class weights + sample weights
# ============================================================

class WeightedSentimentTrainer(Trainer):
    def __init__(
        self,
        *args,
        class_weights: Optional[torch.Tensor] = None,
        label_smoothing: float = 0.0,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.label_smoothing = label_smoothing

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        sample_weight = inputs.pop("sample_weight", None)

        outputs = model(**inputs)
        logits = outputs.logits

        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=weights,
            reduction="none",
            label_smoothing=self.label_smoothing,
        )

        losses = loss_fct(logits, labels)

        if sample_weight is not None:
            sample_weight = sample_weight.to(losses.device).float()
            losses = losses * sample_weight

        loss = losses.mean()

        return (loss, outputs) if return_outputs else loss

In [14]:
# ============================================================
# 13. Modelo y TrainingArguments
# ============================================================

model_config = AutoConfig.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    config=model_config,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

steps_per_epoch = math.ceil(
    len(tokenized_dataset["train"]) / (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
)
total_steps = steps_per_epoch * NUM_EPOCHS
WARMUP_STEPS = int(total_steps * WARMUP_RATIO)

print("steps_per_epoch:", steps_per_epoch)
print("total_steps:", total_steps)
print("warmup_steps:", WARMUP_STEPS)

def build_training_args():
    common_kwargs = dict(
        output_dir=str(MODEL_OUTPUT_DIR),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        logging_steps=25,
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=USE_FP16,
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
    )

    try:
        return TrainingArguments(
            eval_strategy="epoch",
            **common_kwargs,
        )
    except TypeError:
        return TrainingArguments(
            evaluation_strategy="epoch",
            **common_kwargs,
        )

training_args = build_training_args()

trainer = WeightedSentimentTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights=CLASS_WEIGHTS,
    label_smoothing=LABEL_SMOOTHING,
)

print(training_args)

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

steps_per_epoch: 260
total_steps: 1300
warmup_steps: 104
TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
ev

In [15]:
# ============================================================
# 14. Entrenamiento
# ============================================================

train_result = trainer.train()

train_metrics = train_result.metrics

trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))

with (REPORTS_DIR / "sentiment_train_metrics_v1.json").open("w", encoding="utf-8") as f:
    json.dump(train_metrics, f, indent=2, ensure_ascii=False)

print(json.dumps(train_metrics, indent=2, ensure_ascii=False))

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1,Negative Precision,Negative Recall,Negative F1,Negative Support,Neutral Precision,Neutral Recall,Neutral F1,Neutral Support,Positive Precision,Positive Recall,Positive F1,Positive Support
1,1.021960,0.299162,0.926966,0.935576,0.888738,0.909911,0.928053,0.926966,0.925877,0.910891,0.836364,0.872038,110,0.975610,0.851064,0.909091,94,0.920228,0.978788,0.948605,330
2,0.499919,0.267321,0.938202,0.918638,0.916076,0.917220,0.937950,0.938202,0.937991,0.896226,0.863636,0.879630,110,0.895833,0.914894,0.905263,94,0.963855,0.969697,0.966767,330
3,0.401476,0.272508,0.940075,0.919295,0.917086,0.917666,0.940410,0.940075,0.939933,0.922330,0.863636,0.892019,110,0.868687,0.914894,0.891192,94,0.966867,0.972727,0.969789,330
4,0.356019,0.264181,0.947566,0.930953,0.928691,0.929812,0.947461,0.947566,0.947507,0.909091,0.909091,0.909091,110,0.913978,0.904255,0.909091,94,0.969789,0.972727,0.971256,330
5,0.328536,0.260411,0.951311,0.942018,0.930711,0.936195,0.951075,0.951311,0.951078,0.917431,0.909091,0.913242,110,0.944444,0.904255,0.923913,94,0.964179,0.978788,0.971429,330


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "train_runtime": 341.9826,
  "train_samples_per_second": 60.749,
  "train_steps_per_second": 1.901,
  "total_flos": 1113588484635816.0,
  "train_loss": 0.7119096770653358,
  "epoch": 5.0
}


In [16]:
# ============================================================
# 15. Evaluación validation/test
# ============================================================

validation_metrics = trainer.evaluate(tokenized_dataset["validation"])
test_metrics = trainer.evaluate(tokenized_dataset["test"])

print("Validation metrics:")
print(json.dumps(validation_metrics, indent=2, ensure_ascii=False))

print("\nTest metrics:")
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))

metrics_report = {
    "config": CONFIG,
    "column_mapping": column_mapping,
    "split_summary": split_summary,
    "train_metrics": train_metrics,
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
    "label_mapping": {
        "label2id": LABEL2ID,
        "id2label": ID2LABEL,
    },
}

with (REPORTS_DIR / "sentiment_metrics_v1.json").open("w", encoding="utf-8") as f:
    json.dump(metrics_report, f, indent=2, ensure_ascii=False)

Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1,Negative Precision,Negative Recall,Negative F1,Negative Support,Neutral Precision,Neutral Recall,Neutral F1,Neutral Support,Positive Precision,Positive Recall,Positive F1,Positive Support
0.328536,0.260411,5,0.951311,0.942018,0.930711,0.936195,0.951075,0.951311,0.951078,0.917431,0.909091,0.913242,110,0.944444,0.904255,0.923913,94,0.964179,0.978788,0.971429,330


Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1,Negative Precision,Negative Recall,Negative F1,Negative Support,Neutral Precision,Neutral Recall,Neutral F1,Neutral Support,Positive Precision,Positive Recall,Positive F1,Positive Support
0.328536,0.273894,5,0.947162,0.940383,0.930974,0.935334,0.947176,0.947162,0.946953,0.917431,0.934579,0.925926,107,0.945652,0.887755,0.915789,98,0.958065,0.970588,0.964286,306


Validation metrics:
{
  "eval_loss": 0.2604108154773712,
  "eval_accuracy": 0.951310861423221,
  "eval_macro_precision": 0.9420182471942024,
  "eval_macro_recall": 0.9307113690092415,
  "eval_macro_f1": 0.9361945413464174,
  "eval_weighted_precision": 0.9510754933484087,
  "eval_weighted_recall": 0.951310861423221,
  "eval_weighted_f1": 0.9510784188444781,
  "eval_negative_precision": 0.9174311926605505,
  "eval_negative_recall": 0.9090909090909091,
  "eval_negative_f1": 0.91324200913242,
  "eval_negative_support": 110,
  "eval_neutral_precision": 0.9444444444444444,
  "eval_neutral_recall": 0.9042553191489362,
  "eval_neutral_f1": 0.9239130434782609,
  "eval_neutral_support": 94,
  "eval_positive_precision": 0.9641791044776119,
  "eval_positive_recall": 0.9787878787878788,
  "eval_positive_f1": 0.9714285714285714,
  "eval_positive_support": 330
}

Test metrics:
{
  "eval_loss": 0.27389398217201233,
  "eval_accuracy": 0.9471624266144814,
  "eval_macro_precision": 0.9403826275675421,
  

In [17]:
# ============================================================
# 16. Predicciones, probabilidades y matriz de confusión
# ============================================================

def predict_split(split_name: str, base_df: pd.DataFrame) -> pd.DataFrame:
    pred_output = trainer.predict(tokenized_dataset[split_name])
    logits = pred_output.predictions
    probs = softmax_np(logits)
    preds = probs.argmax(axis=-1)

    out = base_df.copy().reset_index(drop=True)
    out["pred_label_id"] = preds
    out["pred_label"] = [ID2LABEL[int(x)] for x in preds]
    out["prob_negative"] = probs[:, LABEL2ID["negative"]]
    out["prob_neutral"] = probs[:, LABEL2ID["neutral"]]
    out["prob_positive"] = probs[:, LABEL2ID["positive"]]
    out["pred_confidence"] = probs.max(axis=-1)
    out["is_correct"] = out["pred_label"] == out["gold_label"]

    return out


val_predictions_df = predict_split("validation", val_df)
test_predictions_df = predict_split("test", test_df)

val_predictions_path = PREDICTIONS_DIR / "sentiment_val_predictions_v1.csv"
test_predictions_path = PREDICTIONS_DIR / "sentiment_test_predictions_v1.csv"

val_predictions_df.to_csv(val_predictions_path, index=False, encoding="utf-8")
test_predictions_df.to_csv(test_predictions_path, index=False, encoding="utf-8")

cm = confusion_matrix(
    test_predictions_df["label_id"],
    test_predictions_df["pred_label_id"],
    labels=[0, 1, 2],
)

cm_df = pd.DataFrame(
    cm,
    index=[f"gold_{x}" for x in LABEL_LIST],
    columns=[f"pred_{x}" for x in LABEL_LIST],
)

cm_path = REPORTS_DIR / "sentiment_test_confusion_matrix_v1.csv"
cm_df.to_csv(cm_path, encoding="utf-8")

print("Confusion matrix:")
display(cm_df)

print("Classification report:")
print(classification_report(
    test_predictions_df["label_id"],
    test_predictions_df["pred_label_id"],
    target_names=LABEL_LIST,
    zero_division=0,
))

display(test_predictions_df[[
    COL_MENTION,
    "gold_label",
    "pred_label",
    "pred_confidence",
    "is_correct",
    "model_input",
]].head(20))

Confusion matrix:


,pred_negative,pred_neutral,pred_positive
gold_negative,100,1,6
gold_neutral,4,87,7
gold_positive,5,4,297


Classification report:
              precision    recall  f1-score   support

    negative       0.92      0.93      0.93       107
     neutral       0.95      0.89      0.92        98
    positive       0.96      0.97      0.96       306

    accuracy                           0.95       511
   macro avg       0.94      0.93      0.94       511
weighted avg       0.95      0.95      0.95       511



,mention_text,gold_label,pred_label,pred_confidence,is_correct,model_input
0,hamburguesa,positive,positive,0.971201,True,Mención: hamburguesa [SEP] Contexto local: Edi...
1,frito variado,negative,positive,0.814496,False,Mención: frito variado [SEP] Contexto local: S...
2,ensaladilla,positive,positive,0.961087,True,Mención: ensaladilla [SEP] Contexto local: Bue...
3,gambas,positive,positive,0.964387,True,Mención: gambas [SEP] Contexto local: Buen sit...
4,tostada con jamon,neutral,neutral,0.967727,True,Mención: tostada con jamon [SEP] Contexto loca...
5,ensaladilla rusa,positive,positive,0.958636,True,Mención: ensaladilla rusa [SEP] Contexto local...
6,tarta de queso,positive,positive,0.969858,True,Mención: tarta de queso [SEP] Contexto local: ...
7,bacalao,positive,positive,0.967304,True,Mención: bacalao [SEP] Contexto local: Resalta...
8,solomillo,positive,positive,0.969187,True,Mención: solomillo [SEP] Contexto local: Resal...
9,arroz con rabo de toro,positive,positive,0.969853,True,Mención: arroz con rabo de toro [SEP] Contexto...


In [18]:
# ============================================================
# 17. Análisis de errores
# ============================================================

error_df = test_predictions_df[test_predictions_df["is_correct"] == False].copy()

error_cols = [
    COL_MENTION,
    COL_DISH_NAME,
    COL_CONTEXT,
    COL_RATING,
    "gold_label",
    "pred_label",
    "pred_confidence",
    "prob_negative",
    "prob_neutral",
    "prob_positive",
    "label_source",
    "model_input",
]

error_cols = [c for c in error_cols if c is not None and c in error_df.columns]

error_analysis_path = PREDICTIONS_DIR / "sentiment_test_error_analysis_v1.csv"
error_df[error_cols].to_csv(error_analysis_path, index=False, encoding="utf-8")

print("Errores test:", error_df.shape)
display(error_df[error_cols].head(50))

print("Errores por gold/pred:")
display(
    error_df.groupby(["gold_label", "pred_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

Errores test: (27, 37)


,mention_text,dish_display_name,context_sentence,rating_value,gold_label,pred_label,pred_confidence,prob_negative,prob_neutral,prob_positive,label_source,model_input
1,frito variado,None,"Sin embargo, le bajo la nota porque no me gust...",4.0,negative,positive,0.814496,0.129566,0.055938,0.814496,weak_hybrid,Mención: frito variado [SEP] Contexto local: S...
10,tataki de atún,None,"El tataki de atún, con ajoblanco sin gluten no...",5.0,negative,positive,0.971862,0.012689,0.015449,0.971862,weak_hybrid,Mención: tataki de atún [SEP] Contexto local: ...
79,tarta,None,Público foto de lo recibido vs lo que tienen e...,2.0,negative,positive,0.533943,0.364774,0.101283,0.533943,weak_hybrid,Mención: tarta [SEP] Contexto local: Público f...
103,calamares,None,Probamos también croquetas de carrillada y de ...,5.0,neutral,positive,0.965934,0.015032,0.019034,0.965934,weak_hybrid,Mención: calamares [SEP] Contexto local: Proba...
104,carrillada,None,Probamos también croquetas de carrillada y de ...,5.0,neutral,positive,0.961600,0.015542,0.022858,0.961600,weak_hybrid,Mención: carrillada [SEP] Contexto local: Prob...
108,croquetas,None,Probamos también croquetas de carrillada y de ...,5.0,neutral,positive,0.960928,0.015848,0.023224,0.960928,weak_hybrid,Mención: croquetas [SEP] Contexto local: Proba...
112,empanadas,None,Queriamos probar la comida colombiana se este ...,4.0,negative,positive,0.706970,0.227777,0.065254,0.706970,weak_hybrid,Mención: empanadas [SEP] Contexto local: Queri...
113,montaditos,None,Los montaditos correctos.,4.0,positive,neutral,0.981209,0.006793,0.981209,0.011998,weak_hybrid,Mención: montaditos [SEP] Contexto local: Los ...
127,gofre,None,"Como podéis ver en las fotos, no he comido nun...",5.0,negative,neutral,0.556748,0.110033,0.556748,0.333220,weak_hybrid,Mención: gofre [SEP] Contexto local: Como podé...
133,tostada de pringá,None,"La tostada de pringá aunque estaba buena, esta...",4.0,negative,positive,0.860728,0.086491,0.052781,0.860728,weak_hybrid,Mención: tostada de pringá [SEP] Contexto loca...


Errores por gold/pred:


,gold_label,pred_label,count
3,neutral,positive,7
1,negative,positive,6
4,positive,negative,5
2,neutral,negative,4
5,positive,neutral,4
0,negative,neutral,1


In [19]:
# ============================================================
# 18. Evaluación por label_source
# ============================================================

def metrics_for_subset(pred_df: pd.DataFrame, subset_name: str) -> Dict[str, Any]:
    if len(pred_df) == 0:
        return {
            "subset": subset_name,
            "rows": 0,
        }

    y_true = pred_df["label_id"].astype(int).values
    y_pred = pred_df["pred_label_id"].astype(int).values

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    return {
        "subset": subset_name,
        "rows": int(len(pred_df)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_macro),
        "macro_recall": float(recall_macro),
        "macro_f1": float(f1_macro),
        "label_distribution": pred_df["gold_label"].value_counts().to_dict(),
    }


group_rows = []

for split_name, pred_df in [
    ("validation", val_predictions_df),
    ("test", test_predictions_df),
]:
    group_rows.append(metrics_for_subset(pred_df, f"{split_name}__all"))

    for source, group in pred_df.groupby("label_source"):
        group_rows.append(metrics_for_subset(group, f"{split_name}__label_source__{source}"))

    if COL_ANNOTATION_STATUS and COL_ANNOTATION_STATUS in pred_df.columns:
        for status, group in pred_df.groupby(COL_ANNOTATION_STATUS):
            group_rows.append(metrics_for_subset(group, f"{split_name}__annotation_status__{status}"))

eval_by_group_df = pd.DataFrame(group_rows)
eval_by_group_path = REPORTS_DIR / "sentiment_eval_by_group_v1.csv"
eval_by_group_df.to_csv(eval_by_group_path, index=False, encoding="utf-8")

display(eval_by_group_df)

,subset,rows,accuracy,macro_precision,macro_recall,macro_f1,label_distribution
0,validation__all,534,0.951311,0.942018,0.930711,0.936195,"{'positive': 330, 'negative': 110, 'neutral': 94}"
1,validation__label_source__manual_gold,227,1.000000,1.000000,1.000000,1.000000,"{'neutral': 81, 'negative': 77, 'positive': 69}"
2,validation__label_source__weak_hybrid,307,0.915309,0.706027,0.659281,0.678425,"{'positive': 261, 'negative': 33, 'neutral': 13}"
3,validation__annotation_status__gold_augmented,227,1.000000,1.000000,1.000000,1.000000,"{'neutral': 81, 'negative': 77, 'positive': 69}"
4,validation__annotation_status__pending,307,0.915309,0.706027,0.659281,0.678425,"{'positive': 261, 'negative': 33, 'neutral': 13}"
5,test__all,511,0.947162,0.940383,0.930974,0.935334,"{'positive': 306, 'negative': 107, 'neutral': 98}"
6,test__label_source__manual_gold,234,0.995726,0.996078,0.995305,0.995664,"{'neutral': 84, 'negative': 79, 'positive': 71}"
7,test__label_source__weak_hybrid,277,0.906137,0.691468,0.643414,0.655214,"{'positive': 235, 'negative': 28, 'neutral': 14}"
8,test__annotation_status__gold_augmented,234,0.995726,0.996078,0.995305,0.995664,"{'neutral': 84, 'negative': 79, 'positive': 71}"
9,test__annotation_status__pending,277,0.906137,0.691468,0.643414,0.655214,"{'positive': 235, 'negative': 28, 'neutral': 14}"


In [20]:
# ============================================================
# 19. Función de inferencia manual
# ============================================================

def predict_sentiment_for_mention(
    mention: str,
    context: str,
    dish_name: str = "",
    rating: Optional[float] = None,
    place_name: str = "",
) -> pd.DataFrame:
    parts = [f"Mención: {mention}"]

    if dish_name and dish_name.lower() != mention.lower():
        parts.append(f"Plato normalizado: {dish_name}")

    if context:
        parts.append(f"Contexto local: {context}")

    if rating is not None:
        parts.append(f"Rating: {rating}")

    if place_name:
        parts.append(f"Local: {place_name}")

    text = " [SEP] ".join(parts)

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    model.eval()
    with torch.no_grad():
        logits = model(**encoded).logits.detach().cpu().numpy()

    probs = softmax_np(logits)[0]

    result = pd.DataFrame({
        "label": LABEL_LIST,
        "probability": probs,
    }).sort_values("probability", ascending=False)

    print("INPUT:")
    print(text)

    return result


examples = [
    ("croquetas", "Las croquetas estaban increíbles, pero el arroz fue decepcionante.", "croquetas", 5),
    ("arroz", "Las croquetas estaban increíbles, pero el arroz fue decepcionante.", "arroz", 5),
    ("pizza", "La pizza no estaba mal, correcta sin más.", "pizza", 3),
    ("tarta de queso", "La tarta de queso fue lo mejor de la cena.", "tarta de queso", 5),
]

for mention, context, dish, rating in examples:
    print("\n========================")
    display(predict_sentiment_for_mention(mention, context, dish, rating))


INPUT:
Mención: croquetas [SEP] Contexto local: Las croquetas estaban increíbles, pero el arroz fue decepcionante. [SEP] Rating: 5


,label,probability
2,positive,0.951141
0,negative,0.024554
1,neutral,0.024304



INPUT:
Mención: arroz [SEP] Contexto local: Las croquetas estaban increíbles, pero el arroz fue decepcionante. [SEP] Rating: 5


,label,probability
0,negative,0.984754
1,neutral,0.010031
2,positive,0.005215



INPUT:
Mención: pizza [SEP] Contexto local: La pizza no estaba mal, correcta sin más. [SEP] Rating: 3


,label,probability
1,neutral,0.987315
0,negative,0.008202
2,positive,0.004483



INPUT:
Mención: tarta de queso [SEP] Contexto local: La tarta de queso fue lo mejor de la cena. [SEP] Rating: 5


,label,probability
2,positive,0.967455
1,neutral,0.016607
0,negative,0.015938


In [21]:
# ============================================================
# 20. Resumen final
# ============================================================

final_summary = {
    "notebook": NOTEBOOK_NAME,
    "version": VERSION,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "model_checkpoint": MODEL_CHECKPOINT,
    "dataset_path": str(DATA_PATH),
    "dataset_rows_raw": int(len(df)),
    "usable_rows": int(len(usable_df)),
    "label_distribution": usable_df["gold_label"].value_counts().to_dict(),
    "label_source_distribution": usable_df["label_source"].value_counts().to_dict(),
    "split_summary": split_summary,
    "train_metrics": train_metrics,
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
    "column_mapping": column_mapping,
    "outputs": {
        "model_dir": str(MODEL_OUTPUT_DIR),
        "metrics_json": str(REPORTS_DIR / "sentiment_metrics_v1.json"),
        "confusion_matrix_csv": str(cm_path),
        "eval_by_group_csv": str(eval_by_group_path),
        "val_predictions_csv": str(val_predictions_path),
        "test_predictions_csv": str(test_predictions_path),
        "test_error_analysis_csv": str(error_analysis_path),
    },
    "notes": [
        "This model performs target/dish-level sentiment classification, not full-review sentiment.",
        "Manual/augmented labels are weighted more than weak hybrid labels.",
        "Macro F1 is the main metric because classes are imbalanced in real reviews.",
        "The model should be applied after mention detection and dish normalization.",
    ],
}

summary_path = REPORTS_DIR / "sentiment_training_summary_v1.json"

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2, ensure_ascii=False)

print(json.dumps(final_summary, indent=2, ensure_ascii=False))

{
  "notebook": "20_train_sevilla_mention_sentiment_absa_v1",
  "version": "sevilla_mention_sentiment_absa_beto_v1",
  "generated_at": "2026-05-14T19:44:24.717137+00:00",
  "model_checkpoint": "dccuchile/bert-base-spanish-wwm-cased",
  "dataset_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-sevilla-v1/sevilla_mention_sentiment_annotation_dataset_v1_extended.jsonl",
  "dataset_rows_raw": 5200,
  "usable_rows": 5200,
  "label_distribution": {
    "positive": 3105,
    "negative": 1104,
    "neutral": 991
  },
  "label_source_distribution": {
    "weak_hybrid": 2911,
    "manual_gold": 2289
  },
  "split_summary": {
    "train_rows": 4155,
    "validation_rows": 534,
    "test_rows": 511,
    "train_labels": {
      "positive": 2469,
      "negative": 887,
      "neutral": 799
    },
    "validation_labels": {
      "positive": 330,
      "negative": 110,
      "neutral": 94
    },
    "test_labels": {
      "positive": 306,
      "negative": 107,
      "neutral": 98
    },

In [22]:
# ============================================================
# 21. Crear ZIP descargable del modelo
# ============================================================

zip_path = OUTPUT_ROOT / f"{VERSION}.zip"

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in MODEL_OUTPUT_DIR.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, file_path.relative_to(MODEL_OUTPUT_DIR.parent))

    for file_path in REPORTS_DIR.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, file_path.relative_to(OUTPUT_ROOT))

    for file_path in PREDICTIONS_DIR.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, file_path.relative_to(OUTPUT_ROOT))

print("ZIP creado:", zip_path)
print("Tamaño MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))

ZIP creado: /kaggle/working/sevilla_mention_sentiment_absa_beto_v1.zip
Tamaño MB: 2423.12
